%md
# 04. Gold – Dimensional Model DDL

## Objetivo

Definir la estructura física del modelo dimensional en la capa Gold, mediante la creación de las tablas de dimensiones y hechos que permitirán organizar y analizar la información de las lecturas IoT.

El modelo sigue un esquema estrella (Star Schema), diseñado para facilitar las consultas analíticas, la generación de indicadores y la integración con herramientas de Business Intelligence.

## Definición de dimensiones

### dim_sensores
Contiene los atributos descriptivos de los sensores, incluyendo su identificador de negocio, tipo, fabricante, fecha de instalación y versión de firmware.

### dim_vehiculos
Almacena los atributos descriptivos de los vehículos, como identificador, marca, modelo, tipo, color, año de fabricación, placa, capacidad de batería y autonomía.

### dim_fecha
Contiene los atributos temporales del calendario, permitiendo analizar las lecturas por día, semana, mes, trimestre y año.

## Definición de la tabla de hechos

### fact_lecturas_iot
Almacena los eventos de lectura generados por los sensores IoT.

Su granularidad corresponde a una fila por cada lectura de un sensor en un instante determinado.

Contiene las claves sustitutas (SK) que relacionan los registros con las dimensiones, las medidas asociadas a las lecturas y los atributos necesarios para su trazabilidad.

## Consideraciones de diseño

- Uso de claves sustitutas (Surrogate Keys) para relacionar las dimensiones con la tabla de hechos.
- Conservación de los identificadores de negocio en las dimensiones.
- Uso de una clave de fecha con formato YYYYMMDD.
- Separación de atributos descriptivos y medidas analíticas.
- Uso de tablas Delta para almacenamiento y gestión de datos.
- Creación de estructuras mediante CREATE TABLE IF NOT EXISTS para preservar los datos existentes durante ejecuciones posteriores.

## Dim _sensores_

In [0]:

CREATE TABLE IF NOT  EXISTS  ecomotive.gold.dim_sensores (


    sk_sensor BIGINT GENERATED ALWAYS AS IDENTITY (start with 1 increment by 1), 
    sensor_id string,
    tipo_sensor string, 
    fabricante string,
    fecha_instalacion timestamp,
    firmware_version string,
     _added_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)

USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults'= 'supported')

## Dim _vehiculos

In [0]:
CREATE TABLE IF NOT EXISTS ecomotive.gold.dim_vehiculos (

    sk_vehiculo BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),

    vehiculo_id STRING,
    marca STRING,
    modelo STRING,
    tipo_vehiculo STRING,
    color STRING,
    anio_fabricacion INT,
    placa STRING,
    capacidad_bateria_kwh DOUBLE,
    autonomia_km DOUBLE,

    _added_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()

)

USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

## DIM FECHA

In [0]:
CREATE TABLE IF NOT EXISTS ecomotive.gold.dim_fecha (

    fecha_key INT,
    fecha DATE,

    dia INT,
    nombre_dia STRING,
    dia_semana INT,

    semana_anio INT,
    mes INT,
    nombre_mes STRING,

    trimestre INT,
    anio INT,

    es_fin_semana BOOLEAN

    
)

USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

## FACT  REGISTROS SENSORES

In [0]:
CREATE TABLE IF NOT EXISTS ecomotive.gold.fact_lecturas_iot(

 row_hash STRING, 
 id_record BIGINT,
 sk_sensor  BIGINT,
 sk_vehiculo BIGINT,
 fecha_key INT,
 momento_de_lectura TIMESTAMP,
 temperatura_c DOUBLE,
 latitud DOUBLE,
 longitud DOUBLE,
 porcentaje_bateria_sensor INT,
 firmware_version STRING,
 signal_strength_dbm DOUBLE,
 estado STRING,
 _added_at TIMESTAMP 
)

USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported' )






